# Project: European Air Traffic Pipeline on EUROCONTROL OPDI data (Medallion Architecture, PySpark)

## DAT535 - Data Engineering

### Thomas Canisares--Morère & Mathis Nancey - Group 08

Batch pipeline that turns EUROCONTROL OPDI flight data into clean, analysis-ready datasets using the Medallion Architecture.

- Period: June 1, 2026 -> July 28, 2026 (~5.5 GB)
- Tables: flight_list (monthly), flight_events and measurements (10-day intervals), OurAirports repositories (airports, runways)

## Part 1: Setup
### Part 1.1: Imports

In [ ]:
import os
import json
import time
import datetime
import random
import csv
import hashlib
import shutil
import urllib.error
import urllib.request
from pathlib import Path
from collections import Counter, defaultdict

import findspark
findspark.init()

from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, lit, when, count, sum as spark_sum, avg,
    min as spark_min, max as spark_max, round as spark_round,
    desc, asc, to_timestamp, to_date, hour, dayofweek,
    lower, upper, trim, concat, coalesce, explode, split,
    countDistinct, collect_list, first, 
    input_file_name, regexp_extract, current_timestamp, broadcast
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, BooleanType, LongType
)

print("Libraries imported successfully")


### Part 1.2: SparkSession

In [ ]:
# SparkSession
spark = SparkSession.builder \
    .appName("DAT535-Project_Thomas_Mathis") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")


### Part 1.3: Parameters

In [ ]:
START = datetime.date(2026, 6, 1)
END   = datetime.date(2026, 7, 29)
ORIGIN     = datetime.date(2022, 1, 1)            # the 10-day periods are counted starting from this date
MIN_FREE_GB = 4.0                                 # minimum free space in GB before downloading a file
RETRIES     = 3                                   # number of download attempts before giving up
PAUSE_S     = 0.5                                 # pause between downloads in seconds

UA = {"User-Agent": "UiS-DAT535-group08"}         # user agent for HTTP requests
BASE_URL   = "https://www.eurocontrol.int/performance/data/download/OPDI/v002"
REF_URLS   = {"airports.csv": "https://davidmegginson.github.io/ourairports-data/airports.csv",
              "runways.csv":  "https://davidmegginson.github.io/ourairports-data/runways.csv"}

PARQUET_TABLES = ["flight_list", "flight_events", "measurements"]

DATA_ROOT      = Path.home() / "opdi-data"
RAW_DIR        = DATA_ROOT / "raw"                # source files as downloaded
BRONZE_DIR     = DATA_ROOT / "bronze"             # raw data + lineage columns
SILVER_DIR     = DATA_ROOT / "silver"
QUARANTINE_DIR = DATA_ROOT / "quarantine"
GOLD_DIR       = DATA_ROOT / "gold"

## Part 2: Bronze Layer
### Part 2.1: Retrieve the data
#### Part 2.1.1: utility functions

In [ ]:
# months() generates the first day of each month in [start, end[
#   - start: the start date (inclusive)
#   - end: the end date (exclusive)
def months(start, end):
    m = datetime.date(start.year, start.month, 1)
    while m < end:
        yield m
        m = datetime.date(m.year + (m.month == 12), m.month % 12 + 1, 1)

# Generate 10-day windows starting from ORIGIN
#   - start: the start date (inclusive)
#   - end: the end date (exclusive)
#   - days: the number of days in each window
def windows(start, end, days=10):
    w = ORIGIN + datetime.timedelta(days=((start - ORIGIN).days // days) * days)
    while w < end:
        yield w, w + datetime.timedelta(days=days)
        w += datetime.timedelta(days=days)

# We return here the HTTP status, the expected size in bytes, and the last modified date (if available)
#   - url: the URL to check
def head(url):
    req = urllib.request.Request(url, method="HEAD", headers=UA)
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            return r.status, int(r.headers.get("Content-Length") or 0), r.headers.get("Last-Modified", "")
    except urllib.error.HTTPError as e:
        return e.code, 0, ""
    except Exception as e:
        return type(e).__name__, 0, ""

# this function downloads the file from the given URL to the destination path.
#   - url: the URL to download from
#   - dest: the destination path (Path object)
def download(url, dest):
    tmp = dest.with_name(dest.name + ".part") # temporary file during download
    for attempt in range(1, RETRIES + 1):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=300) as r, open(tmp, "wb") as f:
                shutil.copyfileobj(r, f, 1 << 20) # copy in 1 MB chunks
            os.replace(tmp, dest)
            return "downloaded"
        except Exception as e:
            print(f"    attempt {attempt}/{RETRIES} failed : {e}")
            tmp.unlink(missing_ok=True)
            time.sleep(5 * attempt)
    return "failed"

# Compute the SHA256 hash of a file at the given path
#   - path: the path to the file (Path object)
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# Return the free disk space in GB for the RAW_DIR
def free_gb():
    return shutil.disk_usage(RAW_DIR).free / 1e9

# Return the current UTC time as a datetime object 
def utc_now():
    return datetime.datetime.now(datetime.timezone.utc)

#### Part 2.1.2: List of files

In [ ]:
files = []
for m in months(START, END):
    n = f"flight_list_{m:%Y%m}.parquet"
    files.append(("flight_list", n, f"{BASE_URL}/flight_list/{n}"))
for table in ("flight_events", "measurements"):
    for w, w_end in windows(START, END):
        n = f"{table}_{w:%Y%m%d}_{w_end:%Y%m%d}.parquet"
        files.append((table, n, f"{BASE_URL}/{table}/{n}"))
for n, url in REF_URLS.items():
    files.append(("reference", n, url))

#### Part 2.1.3: Download

The main loop that downloads the files and records the results in a manifest CSV file.
The manifest contains all the relevant information about each file downloaded and it contains the following columns:
- run_id: a unique identifier for this run of the script
- table: the table name (flight_list, flight_events, measurements, reference)
- file: the file name
- url: the URL from which the file was downloaded
- http_status: the HTTP status code returned by the server
- size_bytes: the size of the file in bytes
- last_modified: the last modified date of the file on the server (if available)
- result: the result of the download (downloaded, already_present, absent, empty, failed)
- sha256: the SHA256 hash of the file
- local_path: the local path of the downloaded file
- retrieved_at_utc: the UTC time when the file was retrieved

In [ ]:
RAW_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = RAW_DIR / "manifest.csv"
run_id = utc_now().strftime("%Y%m%dT%H%M%SZ")
print(f"Period [{START}, {END}[ | {len(files)} files | destination {RAW_DIR} | free {free_gb():.1f} GB\n")
rows, t0 = [], time.time() # list to store the results and the start time
for i, (table, name, url) in enumerate(files, 1):
    status, size, last_modified = head(url)
    dest = RAW_DIR / table / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    if status != 200:
        result = "absent"
    elif size == 0:
        result = "empty"
    elif dest.exists():
        result = "already_present"
    elif free_gb() - size / 1e9 < MIN_FREE_GB:
        print(f"STOP : insufficient free space before {name}")
        break
    else:
        result = download(url, dest)
    ok = result in ("downloaded", "already_present")
    rows.append(dict(run_id=run_id, table=table, file=name, url=url, http_status=status,
                     size_bytes=size, last_modified=last_modified, result=result,
                     sha256=sha256(dest) if ok else "", local_path=str(dest) if ok else "",
                     retrieved_at_utc=utc_now().isoformat(timespec="seconds")))
    print(f"[{i:2d}/{len(files)}] {table:13s} {name:45s} {size/1e6:8.1f} MB  {result}")
    time.sleep(PAUSE_S)

if rows: # write the manifest CSV file
    new = not manifest_path.exists()
    with open(manifest_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        if new:
            w.writeheader()
        w.writerows(rows)

elapsed = time.time() - t0 

#### Part 2.1.4: Summary

In [ ]:
print(f"\nResults : {dict(Counter(r['result'] for r in rows))}") # print summary of results
for t in ("flight_list", "flight_events", "measurements", "reference"): # print summary of files downloaded for each table
    sel = [r for r in rows if r["table"] == t and r["local_path"]]
    print(f"  {t:13s} {len(sel):3d} files  {sum(int(r['size_bytes']) for r in sel)/1e9:6.2f} GB")
print(f"Duration {elapsed:.0f} s | free space left {free_gb():.1f} GB | manifest: {manifest_path}") # duration, free space, manifest path
for r in rows: # print any files that were not downloaded successfully
    if r["result"] not in ("downloaded", "already_present"):
        print(f"  ! {r['file']} -> {r['result']} (HTTP {r['http_status']})")

### Part 2.2: Schema Validation

OPDI files from different periods were observed with different schemas (e.g. `long` vs `decimal(20,0)`
identifiers, dates stored as `string` vs `timestamp`). Before loading anything into Silver, we check that
all the files of a same table share the same schema, and we record each file's schema fingerprint and
row count in `schema_report.csv` for traceability.

In [ ]:
schema_variants = defaultdict(lambda: defaultdict(list))    # group files according to their schema
report_rows = []                                            # one row per file, written to schema_report.csv

for table in PARQUET_TABLES:
    for path in sorted((RAW_DIR / table).glob("*.parquet")):
        df = spark.read.parquet(str(path))                  # reads only the metadata of the Parquet file
        schema = tuple((f.name, f.dataType.simpleString()) for f in df.schema.fields)
        n_rows = df.count()                                
        fingerprint = hashlib.sha256(str(schema).encode()).hexdigest()[:12]
        schema_variants[table][schema].append(path.name)
        report_rows.append(dict(table=table, file=path.name, n_rows=n_rows, n_columns=len(schema),
                                schema_fingerprint=fingerprint,
                                checked_at_utc=utc_now().isoformat(timespec="seconds")))

for table, variants in schema_variants.items():
    keys = list(variants)
    n_files = sum(len(v) for v in variants.values())
    print(f"\n=== {table}: {n_files} files | {len(keys)} schema variant(s) ===")
    for k, key in enumerate(keys, 1):
        names = variants[key]
        listed = ", ".join(names) if len(names) <= 3 else f"{names[0]} ... {names[-1]}"
        print(f"  Variant {k}: {len(names)} file(s) -> {listed}")
    if len(keys) == 1:                                       # single schema: show it
        for name, dtype in keys[0]:
            print(f"      {name:22s} {dtype}")
    else:                                                    # several schemas: show only the differences
        types = [dict(key) for key in keys]
        all_cols = list(dict.fromkeys(c for t in types for c in t))
        print("  Columns that differ between variants:")
        for c in all_cols:
            vals = [t.get(c, "-- missing --") for t in types]
            if len(set(vals)) > 1:
                print(f"      {c:22s} " + " | ".join(f"V{i + 1}: {v}" for i, v in enumerate(vals)))

# Row counts per file
print("\n=== Rows per file ===")
for r in report_rows:
    print(f"  {r['table']:13s} {r['file']:45s} {r['n_rows']:>12,} rows  schema {r['schema_fingerprint']}")

# Save the report
schema_report_path = RAW_DIR / "schema_report.csv"
with open(schema_report_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(report_rows[0]))
    w.writeheader()
    w.writerows(report_rows)
print(f"\nSchema report written to: {schema_report_path}")

### Part 2.3: Bronze tables (row-level lineage)

The source files in `raw/` are kept untouched. The Bronze tables in `bronze/<table>/` contain
exactly the same rows, columns and types, plus row-level lineage columns taken from the manifest:

| Column | Meaning |
|---|---|
| `_source_file` | name of the raw file the row comes from |
| `_source_period` | month (`YYYYMM`) or 10-day window start (`YYYYMMDD`) of the source file — also the partition key |
| `_ingestion_run_id` / `_ingested_at_utc` | download run that retrieved the file, and when |
| `_source_last_modified` | publication date of the file on the EUROCONTROL server |
| `_source_sha256` | fingerprint of the source file |
| `_bronze_run_id` / `_bronze_loaded_at_utc` | run that built this Bronze table, and when |

No cleaning or type conversion happens here. Tables are partitioned by source period, so re-running only
replaces the partitions of the current period. A reconciliation then checks that every raw row is in Bronze.

In [ ]:
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")   # a re-run only replaces the partitions it writes

# 1) File-level lineage: latest successful manifest entry for each file
latest = {}
with open(manifest_path, newline="") as f:
    for r in csv.DictReader(f):
        if r["result"] in ("downloaded", "already_present"):
            latest[r["file"]] = r                                    # chronological file: the last entry wins

lineage_df = spark.createDataFrame(
    [(name, r["run_id"], r["retrieved_at_utc"], r["last_modified"], r["sha256"]) for name, r in latest.items()],
    "_source_file string, _ingestion_run_id string, _ingested_at_utc string, "
    "_source_last_modified string, _source_sha256 string",
).withColumn("_ingested_at_utc", col("_ingested_at_utc").cast("timestamp"))

bronze_run_id = utc_now().strftime("%Y%m%dT%H%M%SZ")

# 2) Row-level lineage: each row gets the name of its source file, then the file metadata from the manifest
def add_lineage(df):
    return (df
        .withColumn("_source_file", regexp_extract(input_file_name(), r"([^/]+)$", 1))
        .withColumn("_source_period", regexp_extract(col("_source_file"), r"_(\d{6,8})(?:_\d{8})?\.parquet$", 1))
        .join(broadcast(lineage_df), "_source_file", "left")            # small table: broadcast join, no shuffle
        .withColumn("_bronze_run_id", lit(bronze_run_id))
        .withColumn("_bronze_loaded_at_utc", current_timestamp()))

# Files of the current period only (list built in 2.1.2)
in_scope = defaultdict(list)
for table, name, _ in files:
    if (RAW_DIR / table / name).exists():
        in_scope[table].append(str(RAW_DIR / table / name))

# 3) OPDI tables: original columns and types unchanged, partitioned by source period
for table in PARQUET_TABLES:
    t_start = time.time()
    (add_lineage(spark.read.parquet(*in_scope[table]))
        .write.mode("overwrite")
        .partitionBy("_source_period")
        .parquet(str(BRONZE_DIR / table)))
    print(f"{table:13s} -> {BRONZE_DIR / table}  ({time.time() - t_start:.0f} s)")

# 4) Reference tables (OurAirports CSV): every column kept as a string, exactly as in the source file
for path in in_scope["reference"]:
    table = Path(path).stem                                            # airports, runways
    raw_ref = spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv(path)
    (add_lineage(raw_ref).drop("_source_period")
        .write.mode("overwrite").parquet(str(BRONZE_DIR / table)))
    print(f"{table:13s} -> {BRONZE_DIR / table}")

In [ ]:
expected = {}                                                          # raw row counts per file
with open(RAW_DIR / "schema_report.csv", newline="") as f:               # written in 2.2 (Parquet files)
    for r in csv.DictReader(f):
        expected[r["file"]] = int(r["n_rows"])
for path in in_scope["reference"]:                                     # CSV files: counted directly
    expected[Path(path).name] = (spark.read.option("header", True).option("multiLine", True)
                                 .option("escape", '"').csv(path).count())

recon = []
for table in PARQUET_TABLES + [Path(p).stem for p in in_scope["reference"]]:
    stats = (spark.read.parquet(str(BRONZE_DIR / table))
             .groupBy("_source_file")
             .agg(count("*").alias("n_rows"),
                  spark_sum(col("_source_sha256").isNull().cast("int")).alias("missing_lineage"))
             .collect())
    for s in stats:
        recon.append(dict(table=table, file=s["_source_file"], raw_rows=expected.get(s["_source_file"]),
                          bronze_rows=s["n_rows"], missing_lineage=s["missing_lineage"]))

print(f"{'table':13s} {'file':45s} {'raw rows':>12s} {'bronze rows':>12s}  status")
for r in sorted(recon, key=lambda r: (r["table"], r["file"])):
    ok = r["raw_rows"] == r["bronze_rows"] and r["missing_lineage"] == 0
    r["status"] = "OK" if ok else "MISMATCH"
    print(f"{r['table']:13s} {r['file']:45s} {r['raw_rows'] or 0:>12,} {r['bronze_rows']:>12,}  {r['status']}")

n_files_expected = sum(len(v) for v in in_scope.values())
all_ok = len(recon) == n_files_expected and all(r["status"] == "OK" for r in recon)
bronze_gb = sum(p.stat().st_size for p in BRONZE_DIR.rglob("*.parquet")) / 1e9
print(f"\nFiles reconciled: {len(recon)}/{n_files_expected} | rows in Bronze: {sum(r['bronze_rows'] for r in recon):,}"
      f" | Bronze size: {bronze_gb:.2f} GB | free space left: {free_gb():.1f} GB")
print("Reconciliation:", "PASSED" if all_ok else "FAILED")

### Part 2.4: Bronze Summary

The Bronze layer successfully loaded raw OPDI datasets (`flight_list`, `flight_events`, `measurements`) and OurAirports reference files (`airports`, `runways`) into structured Parquet storage with zero data loss.

**Key Bronze Layer properties:**
1. **Schema Preservation:** Raw column structures and types are strictly preserved.
2. **Row-level Lineage:** Every single record carries its audit metadata (`_source_file`, `_source_period`, `_ingestion_run_id`, `_ingested_at_utc`, `_source_sha256`, `_bronze_run_id`, `_bronze_loaded_at_utc`).
3. **Audit Reconciliation:** The row counts in Bronze match 100% of the raw data files with zero missing lineage entries.
4. **Partitioning:** Partitioned dynamically by `_source_period` to ensure idempotence during pipeline reruns.


## Part 3: Silver Layer
### Part 3.1: Data understanding

In the Silver layer, our objective is to transform raw Bronze tables into clean, validated, standardized, deduplicated, and enriched datasets ready for analytics and Gold aggregations.

Before implementing transformations, we profile the Bronze tables to evaluate:
- Volume and schema compatibility across tables
- Null rates on key flight attributes (`adep`, `ades`, `first_seen`, `last_seen`, `flt_id`, `typecode`)
- Temporal anomalies (such as inverted timestamps `first_seen > last_seen`)
- Duplicate keys and referential integrity between `flight_events` and `flight_list`


In [ ]:
# -------------------------------------------------------------------------
# Part 3.1: Data Understanding & Profiling of Bronze Tables
# -------------------------------------------------------------------------
from pyspark.sql.functions import (
    col, count, sum as spark_sum, min as spark_min, max as spark_max,
    when, trim, countDistinct
)

print("=" * 70)
print("PART 3.1: PROFILING BRONZE TABLES")
print("=" * 70)

# Load Bronze datasets
bronze_flights_df = spark.read.parquet(str(BRONZE_DIR / "flight_list"))
bronze_events_df = spark.read.parquet(str(BRONZE_DIR / "flight_events"))
bronze_airports_df = spark.read.parquet(str(BRONZE_DIR / "airports"))

flights_count = bronze_flights_df.count()
events_count = bronze_events_df.count()
airports_count = bronze_airports_df.count()

print(f"\n[Volume] Bronze flight_list   : {flights_count:>12,} rows")
print(f"[Volume] Bronze flight_events : {events_count:>12,} rows")
print(f"[Volume] Bronze airports      : {airports_count:>12,} rows")

# Profiling helper: compute null count and percentage for a list of columns
def profile_nulls(df, cols_to_check, total_rows):
    existing = [c for c in cols_to_check if c in df.columns]
    exprs = [spark_sum(when(col(c).isNull() | (trim(col(c).cast("string")) == ""), 1).otherwise(0)).alias(c) for c in existing]
    res = df.select(exprs).collect()[0].asDict()
    print(f"\n--- Null rates for {len(existing)} columns (total rows: {total_rows:,}) ---")
    for c, null_cnt in sorted(res.items(), key=lambda x: x[1], reverse=True):
        pct = (null_cnt / total_rows * 100) if total_rows > 0 else 0.0
        print(f"  {c:25s} : {null_cnt:>10,} nulls ({pct:6.2f}%)")

# Inspect null rates on key flight attributes
profile_nulls(bronze_flights_df, ["id", "flt_id", "icao24", "first_seen", "last_seen", "adep", "ades", "typecode", "model"], flights_count)

# Check for temporal anomalies in flights (first_seen > last_seen)
time_inversions = bronze_flights_df.filter(col("first_seen") > col("last_seen")).count()
print(f"\n[Quality Check] Flights with inverted timestamps (first_seen > last_seen): {time_inversions:,}")

# Check duplicate flight IDs in Bronze
id_col = "id" if "id" in bronze_flights_df.columns else "flight_id"
unique_flight_ids = bronze_flights_df.select(id_col).distinct().count()
duplicate_flight_ids = flights_count - unique_flight_ids
print(f"[Quality Check] Unique flight IDs: {unique_flight_ids:,} | Duplicate IDs in Bronze: {duplicate_flight_ids:,}")

# Events coverage: how many distinct flights in events exist in flight_list
event_flight_col = "flight_id" if "flight_id" in bronze_events_df.columns else "id"
distinct_event_flights = bronze_events_df.select(event_flight_col).distinct()
n_distinct_event_flights = distinct_event_flights.count()
matched_flights = distinct_event_flights.join(bronze_flights_df.select(col(id_col).alias(event_flight_col)), event_flight_col, "inner").count()
print(f"[Referential Integrity] Distinct flights in flight_events: {n_distinct_event_flights:,} (Matched in flight_list: {matched_flights:,} -> {matched_flights / max(n_distinct_event_flights, 1) * 100:.2f}%)")


### Part 3.2: Data Model


The Silver layer defines a clean, relational, type-safe data model optimized for air traffic analytics:

#### 1. `silver_flights` (Consolidated Flights Fact Table)
- **Primary Key:** `flight_id` (`LongType`, cast and unified)
- **Natural Flight Attributes:** `callsign` (broadcast flight number), `transponder_icao24` (24-bit ICAO aircraft hex code)
- **Technical Specs:** `aircraft_typecode` (e.g. `A320`, `B738`), `aircraft_model`, `operator_icao`
- **Standardized Temporal Fields:** `first_seen_utc` (`TimestampType`), `last_seen_utc` (`TimestampType`), `flight_date` (`DateType`), `flight_year` (`IntegerType`), `flight_month` (`IntegerType`)
- **Computed Operational Metrics:** `duration_min` (flight duration in minutes, `DoubleType`), `distance_km` (great-circle route distance in km, `DoubleType`)
- **Origin Aerodrome:** `origin_icao` (4-letter ICAO code), enriched with `origin_airport_name`, `origin_country`, `origin_municipality`, `origin_lat`, `origin_lon`
- **Destination Aerodrome:** `dest_icao` (4-letter ICAO code), enriched with `dest_airport_name`, `dest_country`, `dest_municipality`, `dest_lat`, `dest_lon`
- **Operational Classification:** `route` (e.g. `LFBO-ENZV`), `flight_category` (`DOMESTIC`, `INTERNATIONAL`, `UNKNOWN`)
- **Lineage & Audit:** `_silver_processed_at_utc`, `_source_file`, `_source_period`

#### 2. `silver_flight_events` (Flight Milestones & Trajectory Events)
- **Primary Key:** `event_id` (`StringType`)
- **Foreign Key:** `flight_id` (`LongType`, referencing `silver_flights.flight_id`)
- **Attributes:** `event_type` (`take_off`, `landing`, `climb`, `descent`, `fl_crossing`), `event_timestamp_utc` (`TimestampType`), `latitude` (`DoubleType`), `longitude` (`DoubleType`), `altitude_ft` (`DoubleType`), `info_json` (`StringType`)
- **Audit:** `_silver_processed_at_utc`, `_source_file`, `_source_period`

#### 3. `silver_airports` (Clean Aerodrome Reference Dimension)
- **Primary Key:** `airport_icao` (`StringType`, 4-character ICAO code)
- **Attributes:** `airport_name`, `airport_type`, `iso_country`, `continent`, `municipality`, `latitude`, `longitude`, `elevation_ft`, `iata_code`

#### 4. `quarantine_flights` / `quarantine_flight_events` (Isolated Bad Records)
- Retains original row attributes along with `_quarantine_reason` and `_silver_flagged_at_utc` for auditability and compliance.


### Part 3.3: Standardization


In [ ]:
# -------------------------------------------------------------------------
# Part 3.3: Standardization & Type Casting
# -------------------------------------------------------------------------
from pyspark.sql.functions import (
    col, lit, when, trim, upper, lower, to_timestamp, to_date,
    year, month, hour, dayofweek, concat, coalesce, length,
    unix_timestamp, round as spark_round
)
from pyspark.sql.types import LongType, DoubleType, IntegerType, StringType

print("=" * 70)
print("PART 3.3: STANDARDIZATION & CASTING")
print("=" * 70)

# 1. Standardize Airports reference table
# Keep valid 4-character ICAO codes (ident), trim strings, cast coordinates
std_airports = (bronze_airports_df
    .filter(col("ident").isNotNull() & (length(trim(col("ident"))) == 4))
    .select(
        upper(trim(col("ident"))).alias("airport_icao"),
        trim(col("name")).alias("airport_name"),
        lower(trim(col("type"))).alias("airport_type"),
        upper(trim(col("iso_country"))).alias("iso_country"),
        upper(trim(col("continent"))).alias("continent"),
        trim(col("municipality")).alias("municipality"),
        col("latitude_deg").cast(DoubleType()).alias("latitude"),
        col("longitude_deg").cast(DoubleType()).alias("longitude"),
        col("elevation_ft").cast(DoubleType()).alias("elevation_ft"),
        upper(trim(col("iata_code"))).alias("iata_code")
    )
    .dropDuplicates(["airport_icao"])
)
print(f"[Standardization] Cleaned airports reference: {std_airports.count():,} airports")

# 2. Standardize Flights table
# Dynamically detect columns (handling potential schema variations across periods)
raw_flight_id_col = next((c for c in bronze_flights_df.columns if c.lower() in ("id", "flight_id")), "id")
raw_first_seen    = next((c for c in bronze_flights_df.columns if c.lower() in ("first_seen", "firstseen", "start_time")), "first_seen")
raw_last_seen     = next((c for c in bronze_flights_df.columns if c.lower() in ("last_seen", "lastseen", "end_time")), "last_seen")
raw_adep          = next((c for c in bronze_flights_df.columns if c.lower() in ("adep", "origin", "departure_airport")), "adep")
raw_ades          = next((c for c in bronze_flights_df.columns if c.lower() in ("ades", "destination", "arrival_airport")), "ades")
raw_flt_id        = next((c for c in bronze_flights_df.columns if c.lower() in ("flt_id", "callsign")), "flt_id")
raw_icao24        = next((c for c in bronze_flights_df.columns if c.lower() in ("icao24", "aircraft_address")), "icao24")
raw_typecode      = next((c for c in bronze_flights_df.columns if c.lower() in ("typecode", "aircraft_type")), "typecode")
raw_model         = next((c for c in bronze_flights_df.columns if c.lower() in ("model", "aircraft_model")), "model")
raw_operator      = next((c for c in bronze_flights_df.columns if c.lower() in ("icao_operator", "operator")), "icao_operator")

std_flights = (bronze_flights_df
    .withColumn("flight_id", col(raw_flight_id_col).cast(LongType()))
    .withColumn("callsign", upper(trim(coalesce(col(raw_flt_id) if raw_flt_id in bronze_flights_df.columns else lit(""), lit("")))))
    .withColumn("transponder_icao24", lower(trim(coalesce(col(raw_icao24) if raw_icao24 in bronze_flights_df.columns else lit(""), lit("")))))
    .withColumn("origin_icao", upper(trim(col(raw_adep))))
    .withColumn("dest_icao", upper(trim(col(raw_ades))))
    .withColumn("aircraft_typecode", upper(trim(coalesce(col(raw_typecode) if raw_typecode in bronze_flights_df.columns else lit(""), lit("")))))
    .withColumn("aircraft_model", trim(coalesce(col(raw_model) if raw_model in bronze_flights_df.columns else lit(""), lit(""))))
    .withColumn("operator_icao", upper(trim(coalesce(col(raw_operator) if raw_operator in bronze_flights_df.columns else lit(""), lit("")))))
    # Timestamps & Date extraction (assumes UTC)
    .withColumn("first_seen_utc", to_timestamp(col(raw_first_seen)))
    .withColumn("last_seen_utc", to_timestamp(col(raw_last_seen)))
    .withColumn("flight_date", to_date(col("first_seen_utc")))
    .withColumn("flight_year", year(col("first_seen_utc")))
    .withColumn("flight_month", month(col("first_seen_utc")))
    # Computed duration in minutes
    .withColumn("duration_min", 
                spark_round((unix_timestamp(col("last_seen_utc")) - unix_timestamp(col("first_seen_utc"))) / 60.0, 2))
)

# 3. Standardize Flight Events table
raw_event_fid_col  = next((c for c in bronze_events_df.columns if c.lower() in ("flight_id", "fid")), "flight_id")
raw_event_id_col   = next((c for c in bronze_events_df.columns if c.lower() in ("id", "event_id") and c != raw_event_fid_col), "id")
raw_event_type_col = next((c for c in bronze_events_df.columns if c.lower() in ("type", "event_type")), "type")
raw_event_time_col = next((c for c in bronze_events_df.columns if c.lower() in ("event_time", "timestamp", "time", "event_timestamp")), "event_time")
raw_lat_col        = next((c for c in bronze_events_df.columns if "lat" in c.lower()), "latitude")
raw_lon_col        = next((c for c in bronze_events_df.columns if "lon" in c.lower()), "longitude")
raw_alt_col        = next((c for c in bronze_events_df.columns if "alt" in c.lower()), "altitude")

print(f"[Schema Detection] Events detected columns: flight_id='{raw_event_fid_col}', id='{raw_event_id_col}', type='{raw_event_type_col}', time='{raw_event_time_col}'")

std_events = (bronze_events_df
    .withColumn("flight_id", col(raw_event_fid_col).cast(LongType()))
    .withColumn("event_id", col(raw_event_id_col).cast(StringType()) if raw_event_id_col in bronze_events_df.columns else concat(col(raw_event_fid_col), lit("_"), col(raw_event_time_col)))
    .withColumn("event_type", lower(trim(col(raw_event_type_col))))
    .withColumn("event_timestamp_utc", to_timestamp(col(raw_event_time_col)))
    .withColumn("latitude", col(raw_lat_col).cast(DoubleType()))
    .withColumn("longitude", col(raw_lon_col).cast(DoubleType()))
    .withColumn("altitude_ft", col(raw_alt_col).cast(DoubleType()))
    .withColumn("info_json", col("info") if "info" in bronze_events_df.columns else lit(None).cast(StringType()))
)

print("[Standardization] Formatted flights and events schemas successfully.")


### Part 3.4: Quality and Quarantine rules


In [ ]:
# -------------------------------------------------------------------------
# Part 3.4: Quality & Quarantine Rules
# -------------------------------------------------------------------------
from pyspark.sql.functions import length, when, col, lit, current_timestamp, desc

print("=" * 70)
print("PART 3.4: QUALITY VALIDATION & QUARANTINING")
print("=" * 70)

# Define explicit quality predicates on Flights:
# 1. MISSING_FLIGHT_ID: flight_id must be non-null
# 2. MISSING_TIMESTAMPS: first_seen_utc and last_seen_utc must be valid timestamps
# 3. INVALID_CHRONOLOGY: first_seen_utc must precede or equal last_seen_utc
# 4. NEGATIVE_DURATION: duration must be non-negative
# 5. EXCESSIVE_DURATION_GT_24H: duration must be <= 24 hours (1440 min)
# 6. ORPHAN_ROUTE_NO_VALID_ICAO: both origin and destination ICAO are missing or invalid
flagged_flights = (std_flights
    .withColumn("_quarantine_reason",
        when(col("flight_id").isNull(), lit("MISSING_FLIGHT_ID"))
        .when(col("first_seen_utc").isNull() | col("last_seen_utc").isNull(), lit("MISSING_TIMESTAMPS"))
        .when(col("first_seen_utc") > col("last_seen_utc"), lit("INVALID_CHRONOLOGY"))
        .when(col("duration_min") < 0, lit("NEGATIVE_DURATION"))
        .when(col("duration_min") > 1440, lit("EXCESSIVE_DURATION_GT_24H"))
        .when((col("origin_icao").isNull() | (length(col("origin_icao")) != 4)) &
              (col("dest_icao").isNull() | (length(col("dest_icao")) != 4)), lit("ORPHAN_ROUTE_NO_VALID_ICAO"))
        .otherwise(lit(None))
    )
    .withColumn("_silver_is_valid", col("_quarantine_reason").isNull())
    .withColumn("_silver_flagged_at_utc", current_timestamp())
)

flights_valid = flagged_flights.filter(col("_silver_is_valid") == True)
flights_quarantine = flagged_flights.filter(col("_silver_is_valid") == False)

# Quality predicates on Events:
# 1. MISSING_FLIGHT_ID: must link to a valid flight_id
# 2. MISSING_TIMESTAMP: timestamp must be present and parseable
# 3. INVALID_GPS_COORDINATES: latitude must be in [-90, 90] and longitude in [-180, 180] when non-null
flagged_events = (std_events
    .withColumn("_quarantine_reason",
        when(col("flight_id").isNull(), lit("MISSING_FLIGHT_ID"))
        .when(col("event_timestamp_utc").isNull(), lit("MISSING_TIMESTAMP"))
        .when((col("latitude").isNotNull() & ((col("latitude") < -90.0) | (col("latitude") > 90.0))) |
              (col("longitude").isNotNull() & ((col("longitude") < -180.0) | (col("longitude") > 180.0))),
              lit("INVALID_GPS_COORDINATES"))
        .otherwise(lit(None))
    )
    .withColumn("_silver_is_valid", col("_quarantine_reason").isNull())
    .withColumn("_silver_flagged_at_utc", current_timestamp())
)

events_valid = flagged_events.filter(col("_silver_is_valid") == True)
events_quarantine = flagged_events.filter(col("_silver_is_valid") == False)

# Summary of flight quarantine
total_fl = flagged_flights.count()
valid_fl = flights_valid.count()
quar_fl = flights_quarantine.count()
print(f"Flights Validation - Total: {total_fl:,} | Valid: {valid_fl:,} ({valid_fl/max(total_fl, 1)*100:.2f}%) | Quarantined: {quar_fl:,} ({quar_fl/max(total_fl, 1)*100:.2f}%)")

if quar_fl > 0:
    print("\nQuarantine breakdown for Flights:")
    flights_quarantine.groupBy("_quarantine_reason").count().orderBy(desc("count")).show(truncate=False)

total_ev = flagged_events.count()
valid_ev = events_valid.count()
quar_ev = events_quarantine.count()
print(f"Events Validation  - Total: {total_ev:,} | Valid: {valid_ev:,} ({valid_ev/max(total_ev, 1)*100:.2f}%) | Quarantined: {quar_ev:,} ({quar_ev/max(total_ev, 1)*100:.2f}%)")
if quar_ev > 0:
    print("\nQuarantine breakdown for Events:")
    events_quarantine.groupBy("_quarantine_reason").count().orderBy(desc("count")).show(truncate=False)


### Part 3.5: Deduplication


In [ ]:
# -------------------------------------------------------------------------
# Part 3.5: Deduplication
# -------------------------------------------------------------------------
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

print("=" * 70)
print("PART 3.5: DEDUPLICATION STRATEGY")
print("=" * 70)

# For flights: deduplicate by flight_id keeping the latest ingested record
# (Using _ingested_at_utc or _bronze_loaded_at_utc from Bronze lineage)
order_col = "_ingested_at_utc" if "_ingested_at_utc" in flights_valid.columns else "_bronze_loaded_at_utc"
flight_window = Window.partitionBy("flight_id").orderBy(col(order_col).desc_nulls_last())

flights_deduped = (flights_valid
    .withColumn("_row_num", row_number().over(flight_window))
    .filter(col("_row_num") == 1)
    .drop("_row_num")
)

# For events: deduplicate by (flight_id, event_timestamp_utc, event_type)
event_window = Window.partitionBy("flight_id", "event_timestamp_utc", "event_type").orderBy(col(order_col).desc_nulls_last())

events_deduped = (events_valid
    .withColumn("_row_num", row_number().over(event_window))
    .filter(col("_row_num") == 1)
    .drop("_row_num")
)

flights_before_dedup = flights_valid.count()
flights_after_dedup = flights_deduped.count()
events_before_dedup = events_valid.count()
events_after_dedup = events_deduped.count()

print(f"[Deduplication] Flights: {flights_before_dedup:,} -> {flights_after_dedup:,} (-{flights_before_dedup - flights_after_dedup:,} duplicates)")
print(f"[Deduplication] Events : {events_before_dedup:,} -> {events_after_dedup:,} (-{events_before_dedup - events_after_dedup:,} duplicates)")


### Part 3.6: Enrichment


In [ ]:
# -------------------------------------------------------------------------
# Part 3.6: Dimensional Enrichment & Derived Attributes
# -------------------------------------------------------------------------
from pyspark.sql.functions import (
    broadcast, concat_ws, radians, sin, cos, asin, sqrt,
    current_timestamp, coalesce, when, lit, col, round as spark_round
)
from pyspark.sql.types import DoubleType

print("=" * 70)
print("PART 3.6: DIMENSIONAL ENRICHMENT (OurAirports)")
print("=" * 70)

# Prepare origin aerodrome dimension columns
origin_airports = (std_airports
    .select(
        col("airport_icao").alias("orig_icao_key"),
        col("airport_name").alias("origin_airport_name"),
        col("iso_country").alias("origin_country"),
        col("municipality").alias("origin_municipality"),
        col("latitude").alias("origin_lat"),
        col("longitude").alias("origin_lon")
    )
)

# Prepare destination aerodrome dimension columns
dest_airports = (std_airports
    .select(
        col("airport_icao").alias("dest_icao_key"),
        col("airport_name").alias("dest_airport_name"),
        col("iso_country").alias("dest_country"),
        col("municipality").alias("dest_municipality"),
        col("latitude").alias("dest_lat"),
        col("longitude").alias("dest_lon")
    )
)

# Broadcast join: airports dataset (~70k rows) is small, broadcast prevents full shuffle
flights_enriched = (flights_deduped
    # Join with origin airport
    .join(broadcast(origin_airports), col("origin_icao") == col("orig_icao_key"), "left")
    .drop("orig_icao_key")
    # Join with destination airport
    .join(broadcast(dest_airports), col("dest_icao") == col("dest_icao_key"), "left")
    .drop("dest_icao_key")
    # Route description (e.g. LFBO-ENZV)
    .withColumn("route", concat_ws("-", coalesce(col("origin_icao"), lit("????")), coalesce(col("dest_icao"), lit("????"))))
    # Flight classification (DOMESTIC, INTERNATIONAL, or UNKNOWN)
    .withColumn("flight_category",
        when(col("origin_country").isNull() | col("dest_country").isNull(), lit("UNKNOWN"))
        .when(col("origin_country") == col("dest_country"), lit("DOMESTIC"))
        .otherwise(lit("INTERNATIONAL"))
    )
    # Great-circle distance calculation (Haversine formula in kilometers)
    .withColumn("_dlat", radians(col("dest_lat") - col("origin_lat")))
    .withColumn("_dlon", radians(col("dest_lon") - col("origin_lon")))
    .withColumn("_a", 
        sin(col("_dlat") / 2.0) ** 2 + 
        cos(radians(col("origin_lat"))) * cos(radians(col("dest_lat"))) * (sin(col("_dlon") / 2.0) ** 2)
    )
    .withColumn("distance_km",
        when(col("origin_lat").isNotNull() & col("dest_lat").isNotNull(),
             spark_round(lit(2.0 * 6371.0) * asin(sqrt(col("_a"))), 2))
        .otherwise(lit(None).cast(DoubleType()))
    )
    .drop("_dlat", "_dlon", "_a")
    .withColumn("_silver_processed_at_utc", current_timestamp())
)

# Final selection of clean Silver columns
silver_flights = flights_enriched.select(
    "flight_id", "callsign", "transponder_icao24", "aircraft_typecode", "aircraft_model", "operator_icao",
    "origin_icao", "dest_icao", "route", "flight_category",
    "origin_airport_name", "origin_country", "origin_municipality", "origin_lat", "origin_lon",
    "dest_airport_name", "dest_country", "dest_municipality", "dest_lat", "dest_lon",
    "distance_km", "duration_min", "flight_date", "flight_year", "flight_month",
    "first_seen_utc", "last_seen_utc",
    "_silver_processed_at_utc", "_source_file", "_source_period"
)

silver_events = events_deduped.select(
    "event_id", "flight_id", "event_type", "event_timestamp_utc",
    "latitude", "longitude", "altitude_ft", "info_json",
    current_timestamp().alias("_silver_processed_at_utc"),
    "_source_file", "_source_period"
)

print("[Enrichment] Successfully enriched flights with aerodrome reference data.")
silver_flights.select("flight_id", "callsign", "route", "flight_category", "distance_km", "duration_min").show(5, truncate=False)


### Part 3.7: before-and-after evidence of quality


In [ ]:
# -------------------------------------------------------------------------
# Part 3.7: Before-and-After Quality Evidence Report
# -------------------------------------------------------------------------
from pyspark.sql.functions import count, sum as spark_sum, when, col, desc

print("=" * 70)
print("PART 3.7: DATA QUALITY EVIDENCE (BEFORE VS AFTER)")
print("=" * 70)

bronze_fl_count = bronze_flights_df.count()
silver_fl_count = silver_flights.count()
quar_fl_count = flights_quarantine.count()
dedup_dropped_fl = flights_before_dedup - flights_after_dedup

print("\n--- 1. Pipeline Volume Progression (Flights) ---")
print(f"  Bronze ingested rows        : {bronze_fl_count:>12,}")
print(f"  Quarantined invalid rows    : {quar_fl_count:>12,} ({quar_fl_count / max(bronze_fl_count, 1) * 100:5.2f}%)")
print(f"  Duplicates removed          : {dedup_dropped_fl:>12,} ({dedup_dropped_fl / max(bronze_fl_count, 1) * 100:5.2f}%)")
print(f"  Silver certified valid rows : {silver_fl_count:>12,} ({silver_fl_count / max(bronze_fl_count, 1) * 100:5.2f}%)")

print("\n--- 2. Critical Columns Null Comparison ---")
# Null percentage comparison
raw_nulls = bronze_flights_df.select(
    (spark_sum(when(col("first_seen").isNull(), 1).otherwise(0)) / max(bronze_fl_count, 1) * 100).alias("first_seen"),
    (spark_sum(when(col("last_seen").isNull(), 1).otherwise(0)) / max(bronze_fl_count, 1) * 100).alias("last_seen"),
    (spark_sum(when(col("adep").isNull(), 1).otherwise(0)) / max(bronze_fl_count, 1) * 100).alias("origin_adep"),
    (spark_sum(when(col("ades").isNull(), 1).otherwise(0)) / max(bronze_fl_count, 1) * 100).alias("dest_ades"),
).collect()[0].asDict()

clean_nulls = silver_flights.select(
    (spark_sum(when(col("first_seen_utc").isNull(), 1).otherwise(0)) / max(silver_fl_count, 1) * 100).alias("first_seen"),
    (spark_sum(when(col("last_seen_utc").isNull(), 1).otherwise(0)) / max(silver_fl_count, 1) * 100).alias("last_seen"),
    (spark_sum(when(col("origin_icao").isNull(), 1).otherwise(0)) / max(silver_fl_count, 1) * 100).alias("origin_adep"),
    (spark_sum(when(col("dest_icao").isNull(), 1).otherwise(0)) / max(silver_fl_count, 1) * 100).alias("dest_ades"),
).collect()[0].asDict()

print(f"{'Attribute':20s} {'Bronze Null %':>15s} {'Silver Null %':>15s} Status")
for k in raw_nulls:
    status = 'IMPROVED' if clean_nulls[k] <= raw_nulls[k] else 'EQUAL'
    print(f"{k:20s} {raw_nulls[k]:14.2f}% {clean_nulls[k]:14.2f}%  {status}")

print("\n--- 3. Quarantine Failure Breakdown ---")
if quar_fl_count > 0:
    flights_quarantine.groupBy("_quarantine_reason").count().orderBy(desc("count")).show(truncate=False)
else:
    print("  Zero quarantined flights! All incoming rows passed validation checks.")

print("\n--- 4. Enriched Flight Categories Distribution ---")
silver_flights.groupBy("flight_category").count().orderBy(desc("count")).show(truncate=False)


### Part 3.8: Writing certified data and quarantine data


In [ ]:
# -------------------------------------------------------------------------
# Part 3.8: Writing Silver Certified and Quarantine Datasets
# -------------------------------------------------------------------------
print("=" * 70)
print("PART 3.8: PERSISTENCE (PARQUET)")
print("=" * 70)

# Create Silver and Quarantine directories
SILVER_DIR.mkdir(parents=True, exist_ok=True)
QUARANTINE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Write Silver Flights (Partitioned by flight_year and flight_month for query pruning)
t0 = time.time()
silver_flights_path = SILVER_DIR / "flights"
(silver_flights
    .write.mode("overwrite")
    .partitionBy("flight_year", "flight_month")
    .parquet(str(silver_flights_path)))
print(f"[Storage] silver_flights -> {silver_flights_path} ({time.time() - t0:.1f} s)")

# 2. Write Silver Flight Events (Partitioned by _source_period)
t0 = time.time()
silver_events_path = SILVER_DIR / "flight_events"
(silver_events
    .write.mode("overwrite")
    .partitionBy("_source_period")
    .parquet(str(silver_events_path)))
print(f"[Storage] silver_events  -> {silver_events_path} ({time.time() - t0:.1f} s)")

# 3. Write Silver Airports reference
t0 = time.time()
silver_airports_path = SILVER_DIR / "airports"
(std_airports
    .write.mode("overwrite")
    .parquet(str(silver_airports_path)))
print(f"[Storage] silver_airports-> {silver_airports_path} ({time.time() - t0:.1f} s)")

# 4. Write Quarantine Flights (Partitioned by reason for fast auditing)
if quar_fl_count > 0:
    t0 = time.time()
    quar_path = QUARANTINE_DIR / "flights"
    (flights_quarantine
        .write.mode("overwrite")
        .partitionBy("_quarantine_reason")
        .parquet(str(quar_path)))
    print(f"[Storage] quarantine_flights -> {quar_path} ({time.time() - t0:.1f} s)")

if events_quarantine.count() > 0:
    t0 = time.time()
    quar_ev_path = QUARANTINE_DIR / "flight_events"
    (events_quarantine
        .write.mode("overwrite")
        .partitionBy("_quarantine_reason")
        .parquet(str(quar_ev_path)))
    print(f"[Storage] quarantine_events  -> {quar_ev_path} ({time.time() - t0:.1f} s)")

# Disk footprint
silver_gb = sum(p.stat().st_size for p in SILVER_DIR.rglob("*.parquet")) / 1e9
quar_gb = sum(p.stat().st_size for p in QUARANTINE_DIR.rglob("*.parquet")) / 1e9
print(f"\n[Storage Footprint] Silver size: {silver_gb:.2f} GB | Quarantine size: {quar_gb:.4f} GB | Free space left: {free_gb():.1f} GB")


### Part 3.9: Performance


In [ ]:
# -------------------------------------------------------------------------
# Part 3.9: Spark Execution Performance & Optimizations
# -------------------------------------------------------------------------
print("=" * 70)
print("PART 3.9: PERFORMANCE & OPTIMIZATION EVIDENCE")
print("=" * 70)

# 1. Execution Plan Inspection: BroadcastHashJoin verification
print("\n--- Physical Plan for Enriched Flights (verifying BroadcastHashJoin) ---")
silver_flights.explain(mode="simple")

# 2. Partition Pruning Test
# Querying a specific month from the dataset to demonstrate that only target partitions are read
print("\n--- Partition Pruning Demo ---")
sample_row = spark.read.parquet(str(SILVER_DIR / "flights")).select("flight_year", "flight_month").filter(col("flight_year").isNotNull()).first()
if sample_row:
    target_year, target_month = sample_row["flight_year"], sample_row["flight_month"]
    pruned_df = spark.read.parquet(str(SILVER_DIR / "flights")).filter((col("flight_year") == target_year) & (col("flight_month") == target_month))
    print(f"Target partition scan count ({target_year}-{target_month:02d}): {pruned_df.count():,} rows")

# 3. Memory & Cluster Configurations in use
print("\n--- Active Spark Optimization Configurations ---")
print(f"  spark.sql.adaptive.enabled      : {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"  spark.sql.shuffle.partitions    : {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"  spark.driver.memory             : {spark.conf.get('spark.driver.memory', '2g')}")
print(f"  Disk space remaining on runner  : {free_gb():.1f} GB")
print("\n[SUCCESS] Part 3 (Silver Layer) completed.")


## Part 4: Gold Layer
### Part 4.1: Construction

### Part 4.2: Validation

### Part 4.3: Publication

## Part 5: validation summary

## Part 6: Conclusion and limits